In [ ]:
import sys

import numpy as np
from transforms3d.axangles import axangle2mat, mat2axangle
import torch
import plotly.graph_objects as go
import trimesh

from mano_pybullet.hand_model import HandModel20

In [ ]:
sys.path.append("..")

from utils.grasp_utils import get_handmodel
from model.hand_opt import AdamGraspTransfer

In [ ]:
def mat2rvec(mat):
    """Convert rotation matrix to rotation vector."""
    axis, angle = mat2axangle(mat, unit_thresh=1e-05)
    return axis * angle

def rvec2mat(rvec):
    """Convert rotation vector to rotation matrix."""
    angle = np.linalg.norm(rvec)
    axis = rvec if angle != 0.0 else [0.0, 0.0, 1.0]
    mat = axangle2mat(axis, angle)
    return mat

In [ ]:
# NOTE: Set the mano hand models dir here. When using with a script, load this directory from a some config file

%env MANO_MODELS_DIR=/home/ninad/Projects/MANO/MANO_Hand_Model/mano_v1_2/models

## Set Sample Data

In [ ]:
# Load data for the 00100 frame
# fname = "sample_hamer_output.npz"

# frame_id = "000172"
frame_id = "000247"

fname = f"{frame_id}.npz"


data = np.load(f"../data/{fname}", allow_pickle=True)

In [ ]:
for k in data.keys():
  print(k)

In [ ]:
data['right']

## Set Left/Right

In [ ]:
use_left_hand = True
print("Use left hand? -->", use_left_hand)

rl_index = data['right']
left_idxs = np.arange(rl_index.shape[0])[rl_index==0]
right_idxs = np.arange(rl_index.shape[0])[rl_index==1]

print(left_idxs)
print(right_idxs)

idx_to_use = left_idxs if use_left_hand else right_idxs
print(use_left_hand, idx_to_use)

In [ ]:
print(left_idxs.size, right_idxs.size)

In [ ]:
mano_params = data['pred_mano_params'].item()
print(type(mano_params))
print(mano_params.keys())
print(mano_params['hand_pose'].shape) # for 2 hands

In [ ]:
hand_rotn_mat = mano_params['global_orient'][idx_to_use][0][0]
hand_theta_mat = mano_params['hand_pose'][idx_to_use][0]
mano_trans = data['opt_translation'][idx_to_use][0]
print(hand_rotn_mat.shape)
print(hand_theta_mat.shape)
print(mano_trans.shape)

In [ ]:
hand_theta_full = np.array([mat2rvec(hand_rotn_mat)] + [mat2rvec(hand_theta_mat[i]) for i in range(hand_theta_mat.shape[0])])
print(hand_theta_full.shape)

## Init Gripper Models

In [ ]:
# source_gripper = "mano_left" if use_left_hand else "mano_right"
source_gripper = "mano_right"
target_gripper = "fetch_gripper"
device = "cpu"

In [ ]:
source_model = get_handmodel(
  source_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

sm_left = get_handmodel(
  "mano_left",
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)


In [ ]:
target_model = get_handmodel(
  target_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

## Grasp Pose

In [ ]:
# Mano Pybullet Model
# hand_model = HandModel20(left_hand=use_left_hand)
hand_model = HandModel20(left_hand=False)
hm_left = HandModel20(left_hand=True)

angles, palm_basis = hand_model.mano_to_angles(hand_theta_full)
# angles, _ = hand_model.mano_to_angles(hand_theta_full)
# angles, palm_basis = hm_left.mano_to_angles(hand_theta_full)

angles_left, palm_basis_left = hm_left.mano_to_angles(hand_theta_full)


print(len(angles))
print(palm_basis)

# Reference: https://github.com/kninad/mano_pybullet/blob/960c257cf465f8966e770562b66150beaa359230/mano_pybullet/hand_body.py#L155
origin = hand_model.origins()[0]
# origin = hm_left.origins()[0]

palm_trans = mano_trans + origin - palm_basis @ origin
print(palm_trans.shape, palm_trans)
print(mano_trans)

actual_trans = palm_trans
if use_left_hand:
  actual_trans = palm_trans
  actual_trans -= mano_trans
  actual_trans[0] *= -1
  actual_trans += mano_trans

# palm_trans = np.array([palm_trans[0], mano_trans[1], mano_trans[2]])

In [ ]:
# hand_theta_full[0] = np.zeros(3)

# print(hm_left.mano_to_angles(hand_theta_full)[1])

In [ ]:
# SOURCE GRIPPER (MANO) POSE + DOFS

grasp_pose = torch.zeros(9)
# grasp_pose[0:3] = torch.tensor([0.1, 0.2, 0.3])
# Identity rotation in 6d rot representation is: (1,0,0,0,1,0)
grasp_pose[3:] = torch.tensor(palm_basis.T.reshape(-1)[:6])
grasp_pose[:3] = torch.tensor(actual_trans)
print("Pose:", grasp_pose)

# grasp_dofs = -1 * torch.tensor(angles) if use_left_hand else torch.tensor(angles)
grasp_dofs = torch.tensor(angles)

print("DOFS:", grasp_dofs)

sample_grasp_q = (
  torch.cat(
    [
      grasp_pose,
      grasp_dofs,
    ]
  )
  .unsqueeze(0)
  .to(device)
  .float()
)

In [ ]:
grasp_transfer_opt = AdamGraspTransfer(
  source_gripper,
  target_gripper,
  learning_rate=1e-3,
  device=device
)

In [ ]:
q_traj, energy, _ = grasp_transfer_opt.run_adam(
  sample_grasp_q.squeeze(0), running_name="test"
)

In [ ]:
print(q_traj.shape)
best_q = q_traj[0, -1]
print(best_q.shape)

In [ ]:
if best_q.shape[0] != 9 + len(target_model.dynamic_joints):
  # We optimized only for pose, so need to provide dummy joints
  best_q = torch.cat((best_q, target_model.dynamic_joints_q_upper[0]), dim=0)

## Viz Src + Target

In [ ]:
print("Plotting TARGET and SOURCE together...")

vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red')
target_gripper_mesh_data = target_model.get_plotly_data(q=best_q.unsqueeze(0).float().to(device), color='green')
vis_data += target_gripper_mesh_data
fig = go.Figure(data=vis_data)
fig.show()


In [ ]:
len(target_gripper_mesh_data)

In [ ]:
# trimesh_list = []

# for mesh in target_gripper_mesh_data:
#     vertices = np.array([mesh.x, mesh.y, mesh.z]).T
#     faces = np.array([mesh.i, mesh.j, mesh.k]).T
#     trimesh_list.append(trimesh.Trimesh(vertices=vertices, faces=faces))

# combined_mesh = trimesh.util.concatenate(trimesh_list)
# combined_mesh.export('mesh.ply')

In [ ]:
# q_left = sample_grasp_q.clone()
# q_left[0, 9:] = -1 * torch.tensor(angles_left)


# samp = sample_grasp_q.clone()
# samp[0, :9] = 0
# samp[0, 3] = 1
# samp[0, 7] = 1

# spleft = q_left.clone()
# spleft[0, :9] = 0
# spleft[0, 3] = 1
# spleft[0, 7] = 1

# vis_data = source_model.get_plotly_data(q=samp, color='red', opacity=0.2)
# vis_data += sm_left.get_plotly_data(q=spleft, color='blue', opacity=0.1)

# fig = go.Figure(data=vis_data)
# fig.show()

In [ ]:
l2r_rotmat = axangle2mat(axis=np.array([0, 1, 0]), angle=np.pi)
r2l_rotmat = np.linalg.inv(l2r_rotmat)

l2r_rot6d = l2r_rotmat.T.reshape(-1)[:6]
print(l2r_rot6d.shape)

q_left = sample_grasp_q.clone()
q_left[0, 9:] = -1 * torch.tensor(angles_left)


samp = sample_grasp_q.clone()
# samp[0, :3] = 0
# samp[0, 3] = 1
# samp[0, 7] = 1


# left_rot6d = l2r_rot6d
left_rot6d = (palm_basis @ l2r_rotmat).T.reshape(-1)[:6]
spleft = q_left.clone()
# spleft[0, :3] = 0
spleft[0, 3:9] = torch.tensor(left_rot6d).to(device)



vis_data = source_model.get_plotly_data(q=samp, color='red', opacity=0.2)
vis_data += sm_left.get_plotly_data(q=spleft, color='blue', opacity=0.1)

fig = go.Figure(data=vis_data)
fig.show()

## Viz Mano + URDF

In [ ]:
hand_ply = f"{frame_id}_{int(not use_left_hand)}.ply"
# hand_ply = f"{frame_id}_{1}.ply"

mano_mesh = trimesh.load_mesh(f"../data/{hand_ply}")
print(mano_mesh)

x, y, z = mano_mesh.vertices.T
# i, j, k = mano_mesh.faces.T

vis_data = []
vis_data += source_model.get_plotly_data(q=sample_grasp_q, color='red', opacity=0.5)
# vis_data += target_gripper_mesh_data

# left_rot6d = (palm_basis @ l2r_rotmat).T.reshape(-1)[:6]
# q_left = sample_grasp_q.clone()
# q_left[0, 9:] = -1 * torch.tensor(angles_left)
# q_left[0, 3:9] = torch.tensor(left_rot6d).to(device)
# vis_data += sm_left.get_plotly_data(q=q_left, color='blue', opacity=0.1)

vis_data += [
        go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(size=2, color='green')
        )
    ]


fig = go.Figure(data=vis_data)
fig.show()
# fig.write_html("gtransfer_test.html")


In [ ]:
# verts = np.array(mano_mesh.vertices)
# center = np.mean(verts, axis=0)
# print(center.shape)

# all_verts = verts - mano_trans
# # all_verts[:, 0] *= -1
# # all_verts += mano_trans

# new_grasp = sample_grasp_q.clone()
# new_grasp[0, :3] -= torch.tensor(mano_trans)
# vis_data = source_model.get_plotly_data(q=new_grasp, color='red', opacity=0.2)

# x,y,z = all_verts.T
# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='blue')
#         )
#     ]

# # mano_verts = mano_mesh.vertices
# # mano_verts = verts - mano_trans
# # x,y,z = mano_verts.T
# # vis_data += [
# #         go.Scatter3d(
# #             x=x, y=y, z=z,
# #             mode='markers',
# #             marker=dict(size=2, color='green')
# #         )
# #     ]



# fig = go.Figure(data=vis_data)
# fig.show()




In [ ]:
# mano_trans

In [ ]:
# print(data['right'])
# print(data['opt_translation'][0])
# print(center)


In [ ]:
# verts = np.array(mano_mesh.vertices)
# center = np.mean(verts, axis=0)
# print(center.shape)




# cv = verts - center
# cv[:, 0] *= -1
# # cv[:, 1] *= -1
# nv = cv + center
# x, y, z = nv.T


# all_verts = verts - mano_trans
# all_verts[:, 0] *= -1
# all_verts += mano_trans

# x, y, z = all_verts.T
# # vis_data = []
# vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red', opacity=0.2)


# flip_grasp = sample_grasp_q.clone()
# flip_grasp[0, :3] -= torch.tensor(mano_trans)
# flip_grasp[0, 0] *= -1
# flip_grasp[0, :3] += torch.tensor(mano_trans)

# vis_data = source_model.get_plotly_data(q=flip_grasp, color='orange', opacity=0.5)

# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='blue')
#         )
#     ]

# x,y,z = mano_mesh.vertices.T
# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='green')
#         )
#     ]



# fig = go.Figure(data=vis_data)
# fig.show()
# # fig.write_html("gtransfer_test.html")




In [ ]:
# print(mano_trans)

In [ ]:
# verts = np.array(mano_mesh.vertices)
# center = np.mean(verts, axis=0)
# print(center.shape)

# all_verts = verts - mano_trans
# all_verts[:, 0] *= -1
# # all_verts += mano_trans



# new_grasp = sample_grasp_q.clone()
# new_grasp[0, :3] -= torch.tensor(mano_trans)

# gg_grasp = new_grasp.clone()
# gg_grasp[0, 0] *= -1
# # vis_data = []
# vis_data = source_model.get_plotly_data(q=new_grasp, color='red', opacity=0.2)


# vis_data += source_model.get_plotly_data(q=gg_grasp, color='orange', opacity=0.2)

# x,y,z = all_verts.T
# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='blue')
#         )
#     ]

# mano_verts = mano_mesh.vertices
# mano_verts = verts - mano_trans
# x,y,z = mano_verts.T
# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='green')
#         )
#     ]



# fig = go.Figure(data=vis_data)
# fig.show()
# # fig.write_html("gtransfer_test.html")


